In [ ]:
##Basic QC --- 

In [ ]:
#1.Data Validation

In [ ]:
#1.1 File-level validation

In [ ]:
#1.2 Column-level validation

In [ ]:
#2.Data Verification___Duplicate Check

In [ ]:
#2.1 Duplicate station codes and dates

In [ ]:
#2.2 Check station coordinates

In [ ]:
import pandas as pd
import numpy as np
import re
import glob
import os
from math import radians, sin, cos, asin, sqrt

# ==================== CONFIG ====================
INPUT_GLOB = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-1_TMD_Raw/*.csv"   # <<< เปลี่ยนเป็นโฟลเดอร์ไฟล์รายวันของคุณ
OUT_REPORT_STATION = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-1-1-station_coordinate_qc_report.csv"
OUT_REPORT_DETAILS = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-1-2-station_coordinate_qc_details.csv"
OUT_DUP_PAIRS = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-1-3-station_duplicate_pairs.csv"

# Bounding box ของไทย (ปรับได้ตามพื้นที่ศึกษา)
BBOX = {"lat_min": 5.0, "lat_max": 21.5, "lon_min": 97.0, "lon_max": 106.0}

# เกณฑ์ "ใกล้ซ้อน" สำหรับ Step 4 (เมตร) — ใช้เพื่อ flag REVIEW
NEAR_DUP_THRESHOLD_M = 100
# =================================================


# ----------------- Utilities -----------------
def haversine(lat1, lon1, lat2, lon2):
    """ระยะทางระหว่าง 2 จุด (เมตร) บนทรงกลม"""
    R = 6371000.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * asin(np.sqrt(a))
    return R * c

def looks_like_dms(s):
    """เจอสัญลักษณ์ DMS (°, ′, ″, ', ") หรือรูปแบบ DMS เบื้องต้น"""
    if s is None:
        return False
    s = str(s)
    return bool(re.search(r"[°′'″\"]", s))

def to_numeric_or_nan(x):
    return pd.to_numeric(x, errors="coerce")

def mode_or_first(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    try:
        return s.mode().iloc[0]
    except Exception:
        return s.iloc[0]


# ----------------- Load all CSVs -----------------
files = sorted(glob.glob(INPUT_GLOB))
if not files:
    raise FileNotFoundError(f"No CSVs matched INPUT_GLOB pattern: {INPUT_GLOB}")

dfs = []
for f in files:
    df = pd.read_csv(f)
    # ตรวจคอลัมน์จำเป็น
    required = {"station_id", "station_name_th", "lat", "lon"}
    missing = required - set(df.columns.str.strip())
    if missing:
        raise ValueError(f"{os.path.basename(f)} missing columns: {missing}")

    df["_source_file"] = os.path.basename(f)
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# เก็บค่า raw ไว้ตรวจ DMS
all_df["_raw_lat"] = all_df["lat"].astype(str)
all_df["_raw_lon"] = all_df["lon"].astype(str)

# แปลงเป็นตัวเลขเพื่อเช็ค Step1/2
all_df["_lat"] = to_numeric_or_nan(all_df["lat"])
all_df["_lon"] = to_numeric_or_nan(all_df["lon"])


# ----------------- Step1–Step3 per record -----------------
detail_rows = []  # เก็บรายการผิดเงื่อนไขต่อไฟล์/ต่อวัน

for idx, r in all_df.iterrows():
    sid = r["station_id"]
    sname = r["station_name_th"]
    src = r["_source_file"]
    lat = r["_lat"]
    lon = r["_lon"]
    raw_lat = r["_raw_lat"]
    raw_lon = r["_raw_lon"]

    # Step 1: Range Validation
    if pd.isna(lat) or pd.isna(lon):
        detail_rows.append([sid, sname, src, "Step1", "missing_lat_or_lon", str(raw_lat), str(raw_lon)])
    else:
        if not (-90 <= lat <= 90):
            detail_rows.append([sid, sname, src, "Step1", "lat_out_of_range", lat, lon])
        if not (-180 <= lon <= 180):
            detail_rows.append([sid, sname, src, "Step1", "lon_out_of_range", lat, lon])
        if abs(lat) < 1e-12 and abs(lon) < 1e-12:
            detail_rows.append([sid, sname, src, "Step1", "lat_lon_is_0_0", lat, lon])

    # Step 2: Bounding Box
    if not pd.isna(lat) and not pd.isna(lon):
        in_bbox = (BBOX["lat_min"] <= lat <= BBOX["lat_max"]) and (BBOX["lon_min"] <= lon <= BBOX["lon_max"])
        if not in_bbox:
            detail_rows.append([sid, sname, src, "Step2", "outside_bbox", lat, lon])

    # Step 3: Coordinate System Consistency
    # 3.1 DMS symbols
    if looks_like_dms(raw_lat) or looks_like_dms(raw_lon):
        detail_rows.append([sid, sname, src, "Step3", "has_DMS_symbols", raw_lat, raw_lon])

    # 3.2 magnitude implausible (กันกรณี UTM ฯลฯ)
    if not pd.isna(lat) and not pd.isna(lon):
        if abs(lat) > 200 or abs(lon) > 200:
            detail_rows.append([sid, sname, src, "Step3", "magnitude_implausible", lat, lon])

        # 3.3 สงสัยสลับแกน (ใช้กรอบไทย)
        lat_like_lon = (BBOX["lon_min"] <= lat <= BBOX["lon_max"])
        lon_like_lat = (BBOX["lat_min"] <= lon <= BBOX["lat_max"])
        if lat_like_lon and lon_like_lat:
            detail_rows.append([sid, sname, src, "Step3", "suspect_lat_lon_swapped", lat, lon])

# บันทึกรายละเอียดต่อไฟล์
details_cols = ["station_id", "station_name", "source_file", "step", "issue", "lat_value", "lon_value"]
details_df = pd.DataFrame(detail_rows, columns=details_cols)
details_df.to_csv(OUT_REPORT_DETAILS, index=False)


# ----------------- พิกัดอ้างอิงต่อสถานี -----------------
# ใช้ "mode" ของค่าตัวเลข (หลัง coercion) เป็นพิกัดอ้างอิง
station_ref = (all_df.groupby(["station_id", "station_name_th"], dropna=False)
               .agg(lat_ref=("_lat", mode_or_first),
                    lon_ref=("_lon", mode_or_first),
                    n_records=("station_id", "size"))
               .reset_index())

# ----------------- Step 4: Duplicated Coordinates -----------------
pairs = []
coords = station_ref.dropna(subset=["lat_ref", "lon_ref"]).reset_index(drop=True)
for i in range(len(coords)):
    for j in range(i + 1, len(coords)):
        a = coords.iloc[i]
        b = coords.iloc[j]
        if a.lat_ref == b.lat_ref and a.lon_ref == b.lon_ref:
            pairs.append([a.station_id, b.station_id, "duplicate_exact", 0.0])
        else:
            d = haversine(a.lat_ref, a.lon_ref, b.lat_ref, b.lon_ref)
            if d < NEAR_DUP_THRESHOLD_M:
                pairs.append([a.station_id, b.station_id, "duplicate_near", d])

dup_df = pd.DataFrame(pairs, columns=["station_id_A", "station_id_B", "dup_flag", "distance_m"])
dup_df.to_csv(OUT_DUP_PAIRS, index=False)

# ทำธง duplicates ต่อสถานี
dup_map = {}
if not dup_df.empty:
    for _, r in dup_df.iterrows():
        dup_map.setdefault(r.station_id_A, []).append(f"{r.dup_flag}:{r.station_id_B}@{int(r.distance_m)}m")
        dup_map.setdefault(r.station_id_B, []).append(f"{r.dup_flag}:{r.station_id_A}@{int(r.distance_m)}m")

station_ref["dup_neighbors"] = station_ref["station_id"].map(lambda x: ";".join(dup_map.get(x, [])))
station_ref["dup_flag"] = np.where(station_ref["dup_neighbors"] != "", "has_duplicates", "")

# ----------------- สรุปสถานะ PASS/REVIEW/FAIL ต่อสถานี -----------------
def decide_status(station_id):
    # ถ้าสถานีมีปัญหา Step1/2 ใน "รายละเอียดต่อไฟล์" สักครั้ง ⇒ FAIL
    row_issues = details_df[ (details_df["station_id"] == station_id) & 
                             (details_df["step"].isin(["Step1", "Step2"])) ]
    if not row_issues.empty:
        return "FAIL"

    # ถ้ามี Step3 (swap/DMS/magnitude) ⇒ REVIEW
    row_issues3 = details_df[ (details_df["station_id"] == station_id) & 
                              (details_df["step"] == "Step3") ]
    if not row_issues3.empty:
        return "REVIEW"

    # ถ้ามี duplicates ใกล้ซ้อน ⇒ REVIEW
    if station_id in dup_map:
        return "REVIEW"

    return "PASS"

station_ref["qc_status"] = station_ref["station_id"].apply(decide_status)

# จัดคอลัมน์และบันทึก
station_out = station_ref[["station_id","station_name_th","lat_ref","lon_ref","n_records","dup_flag","dup_neighbors","qc_status"]]
station_out = station_out.sort_values(["qc_status","station_id"])
station_out.to_csv(OUT_REPORT_STATION, index=False)

print(f"Saved:\n - {OUT_REPORT_STATION}\n - {OUT_REPORT_DETAILS}\n - {OUT_DUP_PAIRS}")


In [ ]:
#2.3 Repetition

In [ ]:
import pandas as pd
import numpy as np
import glob, os

# ================== CONFIG ==================
INPUT_GLOB = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-3_Thaiwater_Physical-Limit/*.csv"    # <--- แก้เป็นที่เก็บไฟล์รายวันของคุณ
DATE_COL   = "date"
RAIN_COL   = "rain(mm)"
ID_COLS    = ["station_id","station_name"]  # ต้องมีอย่างน้อย station_id

OUT_RUNS   = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-4-1-nz_constant_runs_flagged.csv"
OUT_DELETED= "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-4-2-nz_constant_deleted_records.csv"
OUT_SUMMARY= "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-4-3-nz_constant_summary_by_file.csv"
OUT_DIR    = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-4_Thaiwater_Repeated"

# เกณฑ์ตามที่กำหนด
THRESH_VAL = 10.0          # ต้องมากกว่า 10 mm/day
MIN_RUNLEN = 5             # ต้อง >= 5 วันติด (มากกว่า 4 วัน)
EPS        = 0.0           # เท่ากัน "เป๊ะ" (ถ้าต้องการยืดหยุ่น ตั้งเป็น 0.1 ได้)
DELETE_KEEP_FIRST = False  # ถ้า True จะ "เก็บวันแรก" แล้วลบตั้งแต่วันที่ 2..L
# ============================================

os.makedirs(OUT_DIR, exist_ok=True)

# 1) โหลดและรวมทุกไฟล์
files = sorted(glob.glob(INPUT_GLOB))
if not files:
    raise FileNotFoundError(f"No files matched: {INPUT_GLOB}")

dfs = []
for f in files:
    df = pd.read_csv(f)
    # ตรวจคอลัมน์จำเป็น
    for c in [DATE_COL, RAIN_COL, "station_id"]:
        if c not in df.columns:
            raise ValueError(f"{os.path.basename(f)} missing required column: {c}")
    df["_source_file"] = os.path.basename(f)
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# 2) เตรียมคอลัมน์ใช้งาน
all_df[DATE_COL] = pd.to_datetime(all_df[DATE_COL], errors="coerce")
all_df["_R"] = pd.to_numeric(all_df[RAIN_COL], errors="coerce")

for c in ID_COLS:
    if c not in all_df.columns:
        all_df[c] = np.nan

# 3) จัดเรียงตามสถานีและเวลา
all_df = all_df.sort_values(["station_id", DATE_COL]).reset_index(drop=True)

# 4) คำนวณ run ของ "ค่าเท่ากัน" รายสถานีแบบวันติดกัน
all_df["_prev_station"] = all_df["station_id"].shift(1)
all_df["_prev_date"]    = all_df[DATE_COL].shift(1)
all_df["_prev_R"]       = all_df["_R"].shift(1)

same_station   = all_df["station_id"].eq(all_df["_prev_station"])
is_consecutive = (all_df[DATE_COL] - all_df["_prev_date"]).dt.days.eq(1)
same_value     = (all_df["_R"].notna() & all_df["_prev_R"].notna() & (all_df["_R"] - all_df["_prev_R"]).abs() <= EPS)

# เริ่ม run ใหม่เมื่อ ไม่ใช่สถานีเดียวกัน หรือไม่ใช่วันถัดไป หรือค่าไม่เท่ากัน
new_run = (~same_station) | (~is_consecutive) | (~same_value)
all_df["_run_id"] = new_run.cumsum()

# 5) สรุป run: ความยาวและค่าที่ซ้ำ
runs = (all_df.groupby(["station_id","_run_id"])
        .agg(run_len=("_R","size"),
             run_value=("_R","first"),
             start_date=(DATE_COL,"first"),
             end_date=(DATE_COL,"last"),
             source_files=("_source_file", lambda x: ",".join(sorted(set(x)))))
        .reset_index())

# เลือกเฉพาะ run ที่เข้าเกณฑ์: ค่ามากกว่า 10 และยาว >= 5
flag_runs = runs[(runs["run_value"] > THRESH_VAL) & (runs["run_len"] >= MIN_RUNLEN)].copy()
flag_runs["rule"]   = f"R>{THRESH_VAL} & run_len>={MIN_RUNLEN}"
flag_runs["action"] = "DELETE"

# 6) ทำเครื่องหมายแถวที่จะถูกลบ
all_df["_delete_flag"] = False

if not flag_runs.empty:
    flag_keys = set(zip(flag_runs["station_id"], flag_runs["_run_id"]))
    in_flag_run = all_df.apply(lambda r: (r["station_id"], r["_run_id"]) in flag_keys, axis=1)

    if DELETE_KEEP_FIRST:
        # เก็บวันแรกของ run ที่ถูก flag แล้วลบวันที่ 2..L
        all_df["_is_first_in_run"] = ~all_df.duplicated(subset=["station_id","_run_id"])
        del_mask = in_flag_run & (~all_df["_is_first_in_run"])
    else:
        # ลบทั้ง run
        del_mask = in_flag_run

    all_df.loc[del_mask, "_delete_flag"] = True

# 7) บันทึกรายงาน run และรายการที่ถูกลบ
flag_runs.sort_values(["station_id","start_date"]).to_csv(OUT_RUNS, index=False)

deleted_rows = all_df[all_df["_delete_flag"]].copy()
keep_cols = [c for c in all_df.columns if not c.startswith("_")] + ["_delete_flag"]
deleted_rows[keep_cols].to_csv(OUT_DELETED, index=False)

# 8) สรุปจำนวนที่ลบต่อไฟล์ และเขียนไฟล์ cleaned รายวัน
summary = []
for f in files:
    base = os.path.basename(f)
    sub = all_df[all_df["_source_file"] == base].copy()
    n_total = len(sub)
    n_del = int(sub["_delete_flag"].sum())

    # เขียนไฟล์ cleaned (ลบแถวที่ถูกลบออก)
    cleaned = sub[~sub["_delete_flag"]].copy()
    drop_cols = [c for c in cleaned.columns if c.startswith("_")]
    cleaned = cleaned.drop(columns=drop_cols)
    cleaned.to_csv(os.path.join(OUT_DIR, base), index=False)

    summary.append({"source_file": base, "n_total": n_total, "n_deleted": n_del, "n_kept": n_total - n_del})

pd.DataFrame(summary).sort_values("source_file").to_csv(OUT_SUMMARY, index=False)

print(f"Saved:\n- {OUT_RUNS}\n- {OUT_DELETED}\n- {OUT_SUMMARY}\n- {OUT_DIR}/*.csv")


In [ ]:
#3.Data Cleansing___Physical Limit Check

In [ ]:
import pandas as pd
import numpy as np
import glob, os

# ================== CONFIG ==================
INPUT_GLOB = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-1_TMD_Raw/*.csv"   # <<< แก้เป็นโฟลเดอร์ไฟล์ของคุณ
RAIN_COL   = "rain(mm)"                   # <<< ชื่อคอลัมน์ปริมาณฝนในไฟล์
OUT_DETAILS = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-2-1-qc_physical_details.csv"   # รายการที่ถูก REVIEW/DELETE ทุกไฟล์รวมกัน
OUT_SUMMARY = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-2-2-qc_physical_summary.csv"   # สรุปต่อไฟล์
OUT_CLEAN_DIR = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-3_Thaiwater_Physical-Limit/" # โฟลเดอร์สำหรับไฟล์ที่ลบ DELETE แล้ว และคง flag ไว้
# เกณฑ์ตามที่กำหนด
REVIEW_MIN = 229.0
HARD_CAP   = 742.0
# ============================================

os.makedirs(OUT_CLEAN_DIR, exist_ok=True)

files = sorted(glob.glob(INPUT_GLOB))
if not files:
    raise FileNotFoundError(f"No files matched: {INPUT_GLOB}")

detail_rows = []
summary_rows = []

for f in files:
    df = pd.read_csv(f)

    if RAIN_COL not in df.columns:
        raise ValueError(f"{os.path.basename(f)} missing column: {RAIN_COL}")

    # แปลงคอลัมน์ฝนเป็นตัวเลข (non-numeric -> NaN)
    r = pd.to_numeric(df[RAIN_COL], errors="coerce")
    df["_R"] = r
    df["_source_file"] = os.path.basename(f)

    # ตั้งค่าเริ่มต้น
    df["_qc_action"] = "KEEP"   # KEEP / REVIEW / DELETE
    df["_qc_reason"] = ""

    # 1) ค่าติดลบ -> DELETE
    mask_neg = df["_R"] < 0
    df.loc[mask_neg, "_qc_action"] = "DELETE"
    df.loc[mask_neg, "_qc_reason"] = "negative_value"

    # 2) > 742 mm/day -> DELETE (ตามสเปก: strictly greater than 742)
    mask_gt_hard = df["_R"] > HARD_CAP
    df.loc[mask_gt_hard, "_qc_action"] = "DELETE"
    df.loc[mask_gt_hard, "_qc_reason"] = f"greater_than_{HARD_CAP}_mm_day"

    # ช่วง 229–742 (inclusive) -> REVIEW (เฉพาะที่ยังไม่ถูกลบ)
    mask_review = (df["_R"] >= REVIEW_MIN) & (df["_R"] <= HARD_CAP) & (df["_qc_action"] == "KEEP")
    df.loc[mask_review, "_qc_action"] = "REVIEW"
    df.loc[mask_review, "_qc_reason"] = f"in_range_{REVIEW_MIN}_to_{HARD_CAP}_mm_day"

    # เก็บรายละเอียด (เฉพาะ REVIEW/DELETE)
    flagged = df[df["_qc_action"].isin(["REVIEW","DELETE"])].copy()
    if not flagged.empty:
        # เก็บคอลัมน์เดิมทั้งหมด + flag/เหตุผล + ไฟล์ต้นทาง
        detail_cols = list(df.columns)
        detail_rows.append(flagged[detail_cols])

    # สรุปต่อไฟล์
    summary_rows.append({
        "source_file": os.path.basename(f),
        "n_total": len(df),
        "n_delete_negative": int(mask_neg.sum()),
        "n_delete_gt_742": int(mask_gt_hard.sum()),
        "n_review_229_to_742": int(mask_review.sum()),
        "n_keep": int((df["_qc_action"] == "KEEP").sum())
    })

    # บันทึกไฟล์ clean (ลบ DELETE ออก แต่คง REVIEW/KEEP พร้อมธง)
    cleaned = df[df["_qc_action"] != "DELETE"].copy()
    # ตัดคอลัมน์ helper ที่ไม่จำเป็นออก (ยกเว้นธง)
    drop_cols = ["_R"]  # คง _qc_action, _qc_reason, _source_file ไว้เพื่อรีวิว
    cleaned = cleaned.drop(columns=drop_cols)
    cleaned.to_csv(os.path.join(OUT_CLEAN_DIR, os.path.basename(f)), index=False)

# รวมและบันทึกรายละเอียด/สรุป
if detail_rows:
    pd.concat(detail_rows, ignore_index=True).to_csv(OUT_DETAILS, index=False)
else:
    pd.DataFrame(columns=["(no REVIEW/DELETE flags)"]).to_csv(OUT_DETAILS, index=False)

pd.DataFrame(summary_rows).to_csv(OUT_SUMMARY, index=False)

print(f"Saved:\n- {OUT_DETAILS}\n- {OUT_SUMMARY}\n- {OUT_CLEAN_DIR}/*.csv")
